# धडा ०७ - नियोजन डिझाईन पॅटर्न

हा नोटबुक Microsoft Agent Framework वापरून AI एजंटसाठी **नियोजन डिझाईन पॅटर्न** चे प्रदर्शन करतो.
तुम्हाला शिकवले जाईल की जटिल प्रवास विनंत्यांना संरचित उपकार्यांमध्ये कसे विभागायचे, त्यांना तज्ञ एजंटना कसे नियुक्त करायचे,
आणि परिणामी योजना कशी अंमलात आणायची — सर्व Pydantic मॉडेल्सद्वारे चालणाऱ्या संरचित आउटपुटचा वापर करून.


## सेटअप


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os, asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Task Decomposition

Task decomposition is the core of the planning design pattern. Instead of asking a single agent to
handle a complex request end-to-end, we break the problem into smaller, well-defined **subtasks**.
Each subtask is assigned to a specialist agent (e.g., flights, hotels, activities) with clear
priorities and dependency ordering.

This approach provides several benefits:
- **Clarity**: each subtask has a single responsibility.
- **Parallelism**: independent subtasks can run concurrently.
- **Reliability**: failures are isolated to individual subtasks.
- **Budget tracking**: costs are estimated per subtask and rolled up.


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## संरचित आउटपुटसह नियोजन एजंट तयार करणे

नियोजन एजंट **फ्रंट डेस्क कोऑर्डिनेटर** म्हणून कार्य करते. उच्च-स्तरीय प्रवास विनंती दिल्यास
तो एक संरचित `TravelPlan` तयार करतो — विनंतीला उपकार्यांमध्ये विभागतो, प्राधान्ये सेट करतो,
आणि अवलंबित्वे ओळखतो जेणेकरून एक कॉनसियरज किंवा अंमलबजावणी स्तर कार्य पार पाडू शकेल.


In [ ]:
planning_agent = client.as_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    options={"response_format": TravelPlan}
)
if result:
    plan = result.value
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## विशेषज्ञ साधनांसह योजना अंमलात आणणे

एकदा फ्रंट डेस्क एजंटने संरचित योजना तयार केल्यावर, **काँसierge एजंट** ती अंमलात आणतो.
प्रत्येक विशेषज्ञ साधन एका उपकामाच्या वर्गाला हाताळते (फ्लाइट्स, हॉटेल्स, क्रियाकलाप). काँसierge
योजना उपकामांमध्ये अवलंबित्व क्रमाने फेरफटका मारतो आणि प्रत्येकाला योग्य साधनावर पाठवतो.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = client.as_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## सारांश

या धड्यात आपण AI एजंटसाठी **योजना डिझाईन पॅटर्न** शिकलात:

1. **कार्य विघटन** — एक फ्रंट डेस्क योजना एजंट एका गुंतागुंतीच्या प्रवास विनंतीला
   संरचित उपकार्यांमध्ये Pydantic मॉडेल्स वापरून विभागतो, प्रत्येक उपकार्याला प्राधान्यक्रम
   आणि अवलंबित्वांसह तज्ञ एजंटला नेमतो.
2. **संरचित आउटपुट** — `response_format` पास केल्याने एजंट मुक्त-रूपातील मजकूराऐवजी
   प्रमाणीकरण झालेल्या `TravelPlan` ऑब्जेक्ट परत करतो, ज्यामुळे पुढील प्रक्रिया विश्वासार्ह होते.
3. **योजना अंमलबजावणी** — एक कन्सीएर्ज एजंट उपकार्यांवर तज्ञ टूल्सचा वापर करून
   (`book_flight`, `reserve_hotel`, `book_activity`) योजना राबवतो आणि निकाल अहवाल करतो.

हा पॅटर्न *काय करायचे* (योजना) आणि *कसे करायचे* (अंमलबजावणी) वेगळे करतो, ज्यामुळे एजंट अधिक
मॅड्युलर, टेस्ट करण्यास सोपे, आणि विस्तारासाठी सुलभ होतात.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
हा दस्तऐवज AI भाषांतर सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) चा वापर करून अनुवादित केला आहे. जरी आम्ही अचूकतेसाठी प्रयत्न करतो, तरी कृपया लक्षात घ्या की स्वयंचलित भाषांतरांमध्ये त्रुटी किंवा अचूकतेची कमतरता असू शकते. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्रोत मानला पाहिजे. महत्त्वाची माहिती असल्यास, व्यावसायिक मानवी भाषांतराची शिफारस केली जाते. या भाषांतराच्या वापरामुळे उद्भवणाऱ्या कोणत्याही गैरसमज किंवा चुकीच्या अर्थलावणीसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
